In [7]:
#importar librerias
import pandas as pd
from google.colab import files
import re

#Cargar archivos desde el computador
uploaded = files.upload()
df1_name = list(uploaded.keys())[0]

uploaded2 = files.upload()
df2_name = list(uploaded2.keys())[0]

uploaded3 = files.upload()
df3_name = list(uploaded.keys())[0]

uploaded4 = files.upload()
df4_name = list(uploaded2.keys())[0]

Saving Migracion Población total.xls to Migracion Población total (1).xls


Saving Crecimiento del PIB anual.xls to Crecimiento del PIB anual (1).xls


Saving Desempleo, total (% de la fuerza laboral total).xls to Desempleo, total (% de la fuerza laboral total).xls


Saving Inflación, precios al consumidor.xls to Inflación, precios al consumidor.xls


In [13]:
# Leer archivos
df1 = pd.read_excel(df1_name, skiprows=3) # Se debe cargar primero el archivo de migración
df2 = pd.read_excel(df2_name, skiprows=3) # se debe cargar luego el archivo de PIB
df3 = pd.read_excel(df3_name, skiprows=3) # archivo de desempleo
df4 = pd.read_excel(df4_name, skiprows=3) # archivo de inflación


#LIMPIAR nombres de columnas con la función
def limpiar_columnas(df):
    nuevas_columnas = {}
    for col in df.columns:
        match = re.search(r'\d{4}', str(col))
        if match:
            nuevas_columnas[col] = match.group()  # deja solo el año
        else:
            nuevas_columnas[col] = col
    return df.rename(columns=nuevas_columnas)


#damos nombres a los df limpios
df1 = limpiar_columnas(df1)
df2 = limpiar_columnas(df2)
df3 = limpiar_columnas(df3)
df4 = limpiar_columnas(df4)


# Filtrar desde 1990 hasta el ultimo año reportado
def filtrar_desde_1990(df):
    columnas_anios = [col for col in df.columns
                      if str(col).isdigit() and int(col) >= 1990]
    return df[['Country Name'] + columnas_anios]

df1 = filtrar_desde_1990(df1)
df2 = filtrar_desde_1990(df2)
df3 = filtrar_desde_1990(df3)
df4 = filtrar_desde_1990(df4)


# Renombrar columnas
df1 = df1.rename(columns=lambda x: f"{x}_migracion" if x != 'Country Name' else x)
df2 = df2.rename(columns=lambda x: f"{x}_pib" if x != 'Country Name' else x)
df3 = df3.rename(columns=lambda x: f"{x}_desempleo" if x != 'Country Name' else x)
df4 = df4.rename(columns=lambda x: f"{x}_inflacion" if x != 'Country Name' else x)


# Unir DataFrames
df_temp = pd.merge(df1, df2, on='Country Name', how='inner')
df_temp = pd.merge(df_temp, df3, on='Country Name', how='inner')
df_final = pd.merge(df_temp, df4, on='Country Name', how='inner')


"""cambiar formato de columnas"""

# Migración
df_migracion = df_final.melt(
    id_vars=['Country Name'],
    value_vars=[col for col in df_final.columns if 'migracion' in col],
    var_name='año',
    value_name='migracion'
)

# PIB
df_pib = df_final.melt(
    id_vars=['Country Name'],
    value_vars=[col for col in df_final.columns if 'pib' in col],
    var_name='año',
    value_name='pib'
)

# Desempleo
df_desempleo = df_final.melt(
    id_vars=['Country Name'],
    value_vars=[col for col in df_final.columns if 'desempleo' in col],
    var_name='año',
    value_name='desempleo'
)

# Inflación
df_inflacion = df_final.melt(
    id_vars=['Country Name'],
    value_vars=[col for col in df_final.columns if 'inflacion' in col],
    var_name='año',
    value_name='inflacion'
)

# Limpiar nombre de año
df_migracion['año'] = df_migracion['año'].str.replace('_migracion', '')
df_pib['año'] = df_pib['año'].str.replace('_pib', '')
df_desempleo['año'] = df_desempleo['año'].str.replace('_desempleo', '')
df_inflacion['año'] = df_inflacion['año'].str.replace('_inflacion', '')



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

#revisamos duplicados y retiramos
df_migracion = df_migracion.drop_duplicates(subset=['Country Name','año'])
df_pib = df_pib.drop_duplicates(subset=['Country Name','año'])
df_desempleo = df_desempleo.drop_duplicates(subset=['Country Name','año'])
df_inflacion = df_inflacion.drop_duplicates(subset=['Country Name','año'])

# Unir bases en formato largo
df_largo = pd.merge(df_migracion, df_pib, on=['Country Name', 'año'], how='left')
df_largo = pd.merge(df_largo, df_desempleo, on=['Country Name', 'año'], how='left')
df_largo = pd.merge(df_largo, df_inflacion, on=['Country Name', 'año'], how='left')


# Renombrar columnas
df_largo = df_largo.rename(columns={'Country Name': 'pais'})

# Convertir año a número
df_largo['año'] = df_largo['año'].astype(int)

# Ordenar
df_largo = df_largo.sort_values(by=['pais', 'año'])

#Lista de países de Suramérica
paises_suramerica = [
    "Argentina", "Bolivia", "Brazil", "Chile", "Colombia",
    "Ecuador", "Guyana", "Paraguay", "Peru",
    "Suriname", "Uruguay", "Venezuela"]

#Filtrar DataFrame con los paises de suramerica
df_suramerica = df_largo[df_largo['pais'].isin(paises_suramerica)]

#Exportar a Excel
df_largo.to_excel("base_panel_completa.xlsx", index=False)
df_suramerica.to_excel("base_panel_suramerica.xlsx", index=False)

#Descargar archivos
files.download("base_panel_completa.xlsx")
files.download("base_panel_suramerica.xlsx")